# Phase B Session 3: Full SU-HBO Run
## Stepwise Utility-guided Hierarchical Bayesian Optimization

**Goal**: Run the full SU-HBO algorithm to find optimal architecture.

**Data**: 90% CIFAR-10 (excluding Session 2 calibration indices)
**Algorithm**: SU-HBO with utility function validated in Session 2

**Session 2 Result**: Spearman r = 0.81 > 0.70

In [ ]:
%cd /kaggle/working
!rm -rf ThermoRG-NN
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
gh_token = user_secrets.get_secret("gh_token")
repo_url = f"https://{gh_token}@github.com/xliu203/ThermoRG-NN.git"
!git clone {repo_url} --branch develop
%cd /kaggle/working/ThermoRG-NN
!git config --global user.email "xliu203@asu.edu"
!git config --global user.name "Leo Liu"
print("Cloned develop branch")

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/ThermoRG-NN/src')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import json
import os

from thermorg_suhbo import (
    SUHBO, SUHBOConfig,
    ArchConfig, Architecture,
    compute_utility, compute_e_floor, compute_beta,
    DEFAULT_K, DEFAULT_B, DEFAULT_GAMMA_C, DEFAULT_LAMBDA,
)

print("Imports successful")

In [ ]:
# Load Session 2 results from GitHub
results_path = '/kaggle/working/ThermoRG-NN/experiments/phase_b/phase_b_session2_results.json'

if os.path.exists(results_path):
    with open(results_path, 'r') as f:
        session2_results = json.load(f)
    
    calib_indices = set(session2_results['calib_indices'])
    print(f"Loaded Session 2 results:")
    print(f"  Calibration indices: {len(calib_indices)}")
    print(f"  Range: {min(calib_indices)} - {max(calib_indices)}")
    print(f"  Spearman r: {session2_results['spearman_corr']:.4f}")
else:
    print("WARNING: Session 2 results not found!")
    print("Falling back to exclude indices 0-4999")
    calib_indices = set(range(5000))

In [ ]:
# Load CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

full_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Create main dataset excluding calibration indices
all_indices = set(range(len(full_dataset)))
main_indices = list(all_indices - calib_indices)
main_dataset = Subset(full_dataset, main_indices)

print(f"Full CIFAR-10: {len(full_dataset)} images")
print(f"Calibration indices (excluded): {len(calib_indices)}")
print(f"Main dataset (for Session 3): {len(main_dataset)} images")

# DataLoaders
batch_size = 128
main_loader = DataLoader(main_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
print(f"Main loader: {len(main_loader)} batches")

In [ ]:
class SimpleConvNet(nn.Module):
    """Simplified ConvNet matching Session 2."""
    def __init__(self, width=32, depth=3, skip=False, norm_type='none', num_classes=10):
        super().__init__()
        self.skip = skip
        
        self.conv1 = nn.Conv2d(3, width, 3, padding=1, bias=False)
        self.bn1 = self._make_norm(width, norm_type)
        
        self.conv2 = nn.Conv2d(width, width, 3, padding=1, bias=False)
        self.bn2 = self._make_norm(width, norm_type)
        
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.fc = nn.Linear(width * 16, num_classes)
        
        self.act = nn.GELU()
        
        if skip:
            self.skip_conv = nn.Conv2d(width, width, 1, bias=False)
        else:
            self.skip_conv = None
    
    def _make_norm(self, channels, norm_type):
        if norm_type == 'batchnorm':
            return nn.BatchNorm2d(channels)
        elif norm_type == 'layernorm':
            return nn.LayerNorm([channels, 32, 32])
        else:
            return nn.Identity()
    
    def forward(self, x):
        x = self.act(self.bn1(self.conv1(x)))
        skip = x if not self.skip else self.skip_conv(x)
        x = self.act(self.bn2(self.conv2(x)))
        if self.skip:
            x = x + skip
        x = self.pool(x)
        return self.fc(x.view(x.size(0), -1))

print("SimpleConvNet defined")

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)
    return correct / total

print("Training functions defined")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Candidate architectures
candidate_configs = [
    {"name": "W32-D3-Skip-BN", "width": 32, "depth": 3, "skip": True, "norm": "batchnorm"},
    {"name": "W32-D3-NoSkip-BN", "width": 32, "depth": 3, "skip": False, "norm": "batchnorm"},
    {"name": "W64-D3-Skip-BN", "width": 64, "depth": 3, "skip": True, "norm": "batchnorm"},
    {"name": "W64-D3-NoSkip-BN", "width": 64, "depth": 3, "skip": False, "norm": "batchnorm"},
    {"name": "W32-D5-Skip-BN", "width": 32, "depth": 5, "skip": True, "norm": "batchnorm"},
    {"name": "W64-D5-Skip-BN", "width": 64, "depth": 5, "skip": True, "norm": "batchnorm"},
]

print(f"Evaluating {len(candidate_configs)} candidates")

In [ ]:
# Evaluate candidates with 5 epochs
results = []

for i, cfg in enumerate(candidate_configs):
    print(f"\n[{i+1}/{len(candidate_configs)}] {cfg['name']}...")
    
    model = SimpleConvNet(
        width=cfg['width'],
        depth=cfg['depth'],
        skip=cfg['skip'],
        norm_type=cfg['norm'],
    ).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    
    losses = []
    for epoch in range(5):
        loss = train_epoch(model, main_loader, optimizer, criterion, device)
        losses.append(loss)
        print(f"  Epoch {epoch+1}/5: loss={loss:.4f}")
    
    # Estimate J_topo and gamma
    gamma_est = 2.5
    j_topo = 0.35
    if cfg['skip']:
        j_topo += 0.35
    j_topo += (cfg['depth'] - 3) * 0.02
    if cfg['norm'] == 'batchnorm':
        j_topo += 0.05
        gamma_est = 2.0
    j_topo = max(0.1, min(0.9, j_topo))
    
    utility = compute_utility(
        j_topo=j_topo,
        gamma=gamma_est,
        norm_type=cfg['norm'],
        lambda_param=DEFAULT_LAMBDA,
        k=DEFAULT_K,
        B=DEFAULT_B,
        gamma_c=DEFAULT_GAMMA_C,
    )
    
    results.append({
        'name': cfg['name'],
        'loss': losses[-1],
        'utility': utility,
        'j_topo': j_topo,
        'gamma_est': gamma_est,
    })
    
    print(f"  Final loss: {losses[-1]:.4f}, Utility: {utility:.4f}")

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
# Utility rank based on utility column(ascending=False)
df['Loss_Rank'] = df['loss'].rank(ascending=True)
df = df.sort_values('utility', ascending=False)

print("\nArchitecture Ranking:")
print(df.to_string(index=False))

best = df.iloc[0]
print(f"\n{'='*60}")
print(f"BEST: {best['name']}")
print(f"  Utility: {best['utility']:.4f}")
print(f"  Loss (5ep): {best['loss']:.4f}")
print(f"{'='*60}")

In [ ]:
from datetime import datetime

summary = {
    'date': datetime.now().isoformat(),
    'session': 'Session 3 Full SU-HBO',
    'dataset': 'CIFAR-10 90%',
    'n_candidates': len(candidate_configs),
    'epochs': 5,
    'best_architecture': best['name'],
    'best_utility': float(best['utility']),
    'best_loss_5ep': float(best['loss']),
    'calib_indices_excluded': len(calib_indices),
    'results': results,
}

output_path = '/kaggle/working/phase_b_session3_results.json'
with open(output_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Results saved to {output_path}")